<h1>Chapter 9 - Multimodal Understanding</h1>
<i>Analyzing Images with your Agent.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 9 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma4:e4b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M.gguf", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gpt-oss-20b", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")

## 2 - Adding Multimodal Understanding

The model that we have been using `Gemma 4 E4B` is a multimodal model and is capable of processing images, audio, and video alongside text. In `Ollama` this only requires parsing the `messages` that we have been leveraging in a special way, namely like so:

```json
[
    {
        "role": "user",
        "content": [
                        {
                            "type": "text",
                            "text": "What’s in this image?"
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                            "url": "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
                            }
                        }
                    ]
    }
]
```

Note how the `content` key now how to separate dictionaries, one containing `text` and the other `image_url`. This allows `Ollama` to process the image and feed it to the LLM. This means that we will have to adjust how the message structure is being used, which requires two changes.

* `memory.py` - Add a parameter to add the `"image_url"`
* `agent.py` - Only add the `"image_url"` to memory when the user provides an image url

Let's explore these changes, starting with `memory.py`:

In [ ]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str, image_url: str = None):
        """Add a message to memory."""
        if image_url:
            content = [{"type": "text", "text": content}, {"type": "image_url", "image_url": {"url": image_url}}]
        self.messages.append({"role": role, "content": content})

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

Note how straightforward these changes are, we merely need to update `.add` with the `image_url`. The changes are minimal:

In [ ]:
from illustrated_agents.chapters.ch9 import memory_diff; memory_diff

The changes to the `TinyAgent` are also minimal, we only need to add the `image_url` when the user's task is first created:

In [ ]:
from illustrated_agents.chapters import ch6_skills


class TinyAgent(ch6_skills.TinyAgent): 
    def run(self, task: str, image_url: str = None) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task, image_url=image_url)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            # Reflection step before taking the next action
            if self.reflector.should_reflect(step):
                self.memory.add("user", self.reflector.prompt)

            # Perform a step and check for completion
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

Only two lines of code need to be changed in order to add this multimodal capabilities to your `TinyAgent`:

In [ ]:
from illustrated_agents.chapters.ch9 import tinyagents_diff; tinyagents_diff

## 3 - Running the Multimodal Agent

Now that you have the necessary components, you can create your `TinyAgent` and give it an image to analyze. Here, we provided the cover of "An Illustrated Guide to AI Agents":

In [ ]:

from illustrated_agents import ReAct, Skills, Reflector, Tools


# Add all modules
tools = Tools()  # No tools
memory = Memory()
react = ReAct(max_steps=10)
reflector = Reflector(interval=5)
skills = Skills()  # No skills

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react, reflector=reflector, skills=skills)

# Run agent on query with image
query = "You do not need tools or skills to view the image. Which animal is on the cover of 'An Illustrated Guide to AI Agents'?"
image_url = "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
print(agent.run(query, image_url=image_url))

The answer is correct, let's see how the model got to that conclusion:

In [ ]:
agent.memory.get_messages()

The behavior of the Agent is quite interesting as it might attempt to use a Tool or Skill to access the image even though it has already "seen" it. This is also why we added the `"You do not need tools or skills to view this image."` in there. Especially smaller models have a tendency to not be aware of their own capabilities. You are unlikely to have this issue when using a model like `Gemini 3`. 

# ▂▂▂▂▂▂▂▂▂▂▂▂